# 00 Getting started

Read-only exploration of the lakehouse with [Polars](https://docs.pola.rs/). Every path comes from `uk_crime.config`, and nothing here writes to `data/`.

Start it with `make explore`. The tables must exist first (`make pipeline`).

In [1]:
import polars as pl

from uk_crime import config

TABLES = {
    "bronze/crimes_street": config.BRONZE_STREET_TABLE,
    "silver/crimes_street": config.SILVER_STREET_TABLE,
    "silver/crimes_street_quarantine": config.SILVER_QUARANTINE_TABLE,
    **{
        f"gold/{p.name}": p
        for p in sorted(config.GOLD_DIR.iterdir())
        if (p / "_delta_log").is_dir()
    },
}
pl.Config.set_tbl_rows(20)
len(TABLES)

14

## Open every Delta table

`pl.scan_delta` reads the Delta log through `deltalake` and returns a lazy frame, so counting rows does not load the data.

In [2]:
for name, path in TABLES.items():
    lazy = pl.scan_delta(str(path))
    rows = lazy.select(pl.len()).collect().item()
    schema = lazy.collect_schema()
    columns = ", ".join(f"{c}:{t}" for c, t in schema.items())
    print(f"{name}: {rows:,} rows, {len(schema)} columns\n    {columns}")

bronze/crimes_street: 3,440,762 rows, 18 columns
    crime_id:String, month:String, reported_by:String, falls_within:String, longitude:String, latitude:String, location:String, lsoa_code:String, lsoa_name:String, crime_type:String, last_outcome_category:String, context:String, _corrupt_record:String, _archive_month:Date, _month:Date, _force:String, _source_file:String, _ingested_at:Datetime(time_unit='us', time_zone='UTC')
silver/crimes_street: 3,416,073 rows, 20 columns
    record_key:String, crime_id:String, month:Date, force:String, reported_by:String, falls_within:String, longitude:Float64, latitude:Float64, location:String, lsoa_code:String, lsoa_name:String, crime_type:String, last_outcome_category:String, context:String, is_asb:Boolean, has_location:Boolean, dq_attribute_conflict:Boolean, _archive_month:Date, _source_file:String, _run_id:String
silver/crimes_street_quarantine: 1,279 rows, 21 columns
    crime_id:String, month:String, reported_by:String, falls_within:String, long

## Reproduce the Criminal damage and arson drop

Metropolitan Police monthly counts of `Criminal damage and arson` in silver. The count falls from about 4,300 in 2026-04 to about 220 in 2026-05 and stays there. No other crime type moves by more than 50% in a month.

In [3]:
silver = pl.scan_delta(str(config.SILVER_STREET_TABLE))
(
    silver.filter(
        pl.col("force") == "metropolitan",
        pl.col("crime_type") == "Criminal damage and arson",
        pl.col("month").is_between(pl.date(2025, 9, 1), pl.date(2026, 8, 1)),
    )
    .group_by("month")
    .agg(pl.len().alias("crimes"))
    .sort("month")
    .collect()
)

month,crimes
date,u32
2025-09-01,4505
2025-10-01,4659
2025-11-01,4435
2025-12-01,4221
2026-01-01,4222
2026-02-01,3978
2026-03-01,4405
2026-04-01,4322
2026-05-01,221


## Polars for PySpark users

The same aggregation both ways: non-ASB crimes per crime type in the newest month, Metropolitan only. PySpark (shown, not run here, so no Spark session is started):

```python
from pyspark.sql import functions as F

(
    spark.read.format("delta").load(str(config.SILVER_STREET_TABLE))
    .where((F.col("force") == "metropolitan") & ~F.col("is_asb")
           & (F.col("month") == "2026-08-01"))
    .groupBy("crime_type")
    .agg(F.count("*").alias("crimes"))
    .orderBy(F.desc("crimes"))
    .show()
)
```

Polars: `where` is `filter`, `groupBy` is `group_by`, `agg` takes expressions, and `F.col` becomes `pl.col`. A `LazyFrame` plans like a Spark DataFrame until `collect()` runs it.

In [4]:
(
    silver.filter(
        (pl.col("force") == "metropolitan")
        & ~pl.col("is_asb")
        & (pl.col("month") == pl.date(2026, 8, 1))
    )
    .group_by("crime_type")
    .agg(pl.len().alias("crimes"))
    .sort("crimes", descending=True)
    .collect()
)

crime_type,crimes
str,u32
"""Violence and sexual offences""",24374
"""Shoplifting""",9336
"""Other theft""",8283
"""Vehicle crime""",6989
"""Public order""",5317
"""Theft from the person""",4665
"""Drugs""",4191
"""Burglary""",3916
"""Robbery""",2479


# Open questions

Empty cells for your own exploration.

## Borough charge rates and crime mix

## Bicycle-theft seasonality

## Robbery hotspots

## City of London vs Westminster